# AI@UCI — Winter Week 4: Decision Trees & Ensembling
### From Questions to Code

**Goal:** make the logic from the slides executable.

> **data → try splits → measure impurity → choose a split → grow a tree → control depth → combine trees → test**

We will:
- train and inspect one decision tree,
- connect splits to Gini impurity,
- see how tree depth changes generalization,
- visualize decision regions,
- and compare one tree with a random forest.

## 0. Setup

We use:
- `numpy` — arrays and grids
- `pandas` — readable tables
- `matplotlib` — visualizing data and decision regions
- `scikit-learn` — datasets, decision trees, random forests, train/test split, and accuracy

The plotting helpers are mostly completed so the live activity stays focused on **tree logic**, not plotting syntax.

In [ ]:
# %pip install -q numpy pandas matplotlib scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42

## 1. What if one line is not enough?

The slides started with curved regions that are awkward for one straight boundary.

`make_moons` gives us a small two-feature classification problem with that shape.

In [ ]:
X, y = make_moons(
    n_samples=240,
    noise=0.28,
    random_state=RANDOM_STATE
)

df = pd.DataFrame(X, columns=["feature_1", "feature_2"])
df["label"] = y

df.head()

### Completed example: visualize the feature space

Before training anything, look at the geometry.

**Question:** could one straight line perfectly separate these classes?

In [ ]:
for label, group in df.groupby("label"):
    plt.scatter(
        group["feature_1"],
        group["feature_2"],
        label=f"Class {label}",
        alpha=0.75
    )

plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.title("A Nonlinear Classification Problem")
plt.legend()
plt.show()

## 2. Split before learning

As in the earlier workshops:
- the **training set** is used to learn the model,
- the **test set** stays unseen until evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("training examples:", len(X_train))
print("test examples:", len(X_test))

## 3. Gini impurity: how mixed is a group?

For class proportions \(p_1, p_2, \dots\):

\[
Gini = 1 - \sum_i p_i^2
\]

A pure group has Gini \(= 0\).  
A more mixed group has a larger Gini score.

### Completed example: calculate Gini for one group

Suppose a node contains 4 red examples and 1 blue example.

In [ ]:
red = 4
blue = 1
total = red + blue

p_red = red / total
p_blue = blue / total

gini = 1 - (p_red ** 2 + p_blue ** 2)

print("Gini:", round(gini, 3))

### Your turn: calculate Gini for a 3-vs-3 group

In [ ]:
red = 3
blue = 3

total = red + blue
p_red = red / total
p_blue = blue / total
gini = 1 - (p_red ** 2 + p_blue ** 2)

print("Gini:", round(gini, 3))

## 4. Train one decision tree

Start with a **shallow tree** so the learned rules are easy to inspect.

### Completed example: a depth-2 tree

In [ ]:
shallow_tree = DecisionTreeClassifier(
    criterion="gini",
    max_depth=2,
    random_state=RANDOM_STATE
)

shallow_tree.fit(X_train, y_train)

print("train accuracy:", round(shallow_tree.score(X_train, y_train), 3))
print("test accuracy:", round(shallow_tree.score(X_test, y_test), 3))

## 5. Inspect the learned questions

A tree is a learned chain of if/else questions.

`export_text` lets us print those questions directly.

In [ ]:
print(
    export_text(
        shallow_tree,
        feature_names=["feature_1", "feature_2"]
    )
)

### Completed visualization: see the tree structure

In [ ]:
plt.figure(figsize=(12, 6))
plot_tree(
    shallow_tree,
    feature_names=["feature_1", "feature_2"],
    class_names=["Class 0", "Class 1"],
    filled=True,
    rounded=True
)
plt.title("A Shallow Decision Tree")
plt.show()

## 6. When does a tree overfit?

More depth means more flexibility.

A deep tree can keep splitting until it models tiny details in the training set.

In [ ]:
deep_tree = DecisionTreeClassifier(
    criterion="gini",
    max_depth=8,
    random_state=RANDOM_STATE
)

deep_tree.fit(X_train, y_train)

shallow_train = shallow_tree.score(X_train, y_train)
shallow_test = shallow_tree.score(X_test, y_test)

deep_train = deep_tree.score(X_train, y_train)
deep_test = deep_tree.score(X_test, y_test)

print("shallow tree:", round(shallow_train, 3), round(shallow_test, 3))
print("deep tree:   ", round(deep_train, 3), round(deep_test, 3))

## 7. Compare many tree depths

This cell is completed so we can inspect the pattern quickly.

Look for the point where training performance keeps improving but test performance stops improving.

In [ ]:
depths = range(1, 16)
train_scores = []
test_scores = []

for depth in depths:
    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=RANDOM_STATE
    )
    model.fit(X_train, y_train)

    train_scores.append(model.score(X_train, y_train))
    test_scores.append(model.score(X_test, y_test))

plt.plot(depths, train_scores, marker="o", label="train")
plt.plot(depths, test_scores, marker="o", label="test")
plt.xlabel("max_depth")
plt.ylabel("accuracy")
plt.title("Tree Depth vs. Accuracy")
plt.legend()
plt.show()

## 8. Visualize decision regions

A decision tree divides feature space into axis-aligned regions.

The helper below is already complete.

In [ ]:
def plot_decision_regions(model, X, y, title):
    x1_min, x1_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    x2_min, x2_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

    xx, yy = np.meshgrid(
        np.linspace(x1_min, x1_max, 300),
        np.linspace(x2_min, x2_max, 300)
    )

    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = model.predict(grid).reshape(xx.shape)

    plt.figure(figsize=(7, 5))
    plt.contourf(xx, yy, zz, alpha=0.25)
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolor="k", alpha=0.8)
    plt.xlabel("feature 1")
    plt.ylabel("feature 2")
    plt.title(title)
    plt.show()


plot_decision_regions(shallow_tree, X_train, y_train, "Shallow Tree Decision Regions")
plot_decision_regions(deep_tree, X_train, y_train, "Deep Tree Decision Regions")

## 9. One tree can be fragile

A single tree can change when the training sample changes.

Random forests reduce that instability by combining many different trees.

### Completed example: one tree's prediction

In [ ]:
point = np.array([[0.5, 0.2]])

single_prediction = deep_tree.predict(point)[0]

print("single tree prediction:", single_prediction)

## 10. Random forest: many different trees vote

Each tree gets a slightly different view of the training problem.

The forest combines their predictions.

In [ ]:
forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    random_state=RANDOM_STATE
)

forest.fit(X_train, y_train)

forest_train = forest.score(X_train, y_train)
forest_test = forest.score(X_test, y_test)

print("random forest train accuracy:", round(forest_train, 3))
print("random forest test accuracy:", round(forest_test, 3))

## 11. See the vote

This cell is completed.

We ask the first few trees in the forest to predict the same point.

In [ ]:
votes = np.array([
    tree.predict(point)[0]
    for tree in forest.estimators_[:15]
])

print("first 15 tree votes:", votes)
print("forest prediction:", forest.predict(point)[0])

## 12. Compare one tree vs. many trees

In [ ]:
deep_test_accuracy = accuracy_score(y_test, deep_tree.predict(X_test))
forest_test_accuracy = accuracy_score(y_test, forest.predict(X_test))

print("deep tree test accuracy:", round(deep_test_accuracy, 3))
print("random forest test accuracy:", round(forest_test_accuracy, 3))

### Completed visualization: forest decision regions

In [ ]:
plot_decision_regions(
    forest,
    X_train,
    y_train,
    "Random Forest Decision Regions"
)

## 13. Reflection

Make sure you can explain these in plain English:

1. What does one tree split represent?
2. What does Gini impurity measure?
3. Why can a deeper tree overfit?
4. What does `max_depth` control?
5. Why can one decision tree be unstable?
6. Why does a random forest intentionally make its trees different?
7. What is the difference between one tree's prediction and a forest's prediction?

## Optional extension: boosting

The slides briefly introduced another ensemble idea.

- **Random forest / bagging:** train many trees in parallel and combine them.
- **Boosting:** train models sequentially so later models focus more on earlier mistakes.

We will keep the coding activity focused on decision trees and random forests.